In [0]:
from pyspark.sql import functions as F

SILVER = "workspace.s4lake_silver"
GOLD = "workspace.s4lake_gold"
DATA_CORTE = "2026-09-22"

def salvar(df, schema, tabela):
    df.write.mode("overwrite").option("overwriteSchema", True).saveAsTable(f"{schema}.{tabela}")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD}")

df = spark.table(f"{SILVER}.titulos_receber")

fato_titulos = df.withColumn(
    "dias_atraso",
    F.when(
        F.col("status") == "pago",
        F.datediff(F.col("data_pagamento"), F.col("data_vencimento"))
    ).when(
        F.col("status") == "aberto",
        F.datediff(F.lit(DATA_CORTE).cast("date"), F.col("data_vencimento"))
    )
)

fato_titulos = fato_titulos.withColumn(
    "situacao",
    F.when(
        (F.col("status") == "pago") & (F.col("dias_atraso") <= 0),
        F.lit("pago no prazo")
    ).when(
        (F.col("status") == "pago") & (F.col("dias_atraso") > 0),
        F.lit("pago com atraso")
    ).when(
        (F.col("status") == "aberto") & (F.col("dias_atraso") <= 0),        
        F.lit("a vencer")
    ).when(
        (F.col("status") == "aberto") & (F.col("dias_atraso") > 0),        
        F.lit("vencido")
    )
)

fato_titulos = fato_titulos.withColumn(
    "faixa_aging",
    F.when(
        (F.col("status") == "aberto") & (F.col("dias_atraso") <= 0),
        F.lit("a vencer")
    ).when(
        (F.col("status") == "aberto") & (F.col("dias_atraso").between(1, 30)),
        F.lit("1-30")
    ).when(
        (F.col("status") == "aberto") & (F.col("dias_atraso").between(31, 60)),
        F.lit("31-60")
    ).when(
        (F.col("status") == "aberto") & (F.col("dias_atraso").between(61, 90)),
        F.lit("61-90")
    ).when(
        (F.col("status") == "aberto") & (F.col("dias_atraso") > 90),
        F.lit("90+")
    )
)

fato_titulos = fato_titulos.withColumn(
    "data_corte", F.lit(DATA_CORTE).cast("date")
)

salvar(fato_titulos, GOLD, "fato_titulos")

spark.sql(f"""
    SELECT
        (SELECT count(*) FROM {GOLD}.fato_titulos) AS total_linhas,
        (SELECT count(*) FROM {GOLD}.fato_titulos WHERE dias_atraso IS NULL) AS atraso_nulo,
        (SELECT count(*) FROM {GOLD}.fato_titulos WHERE situacao IS NULL) AS situacao_nula
""").show()

In [0]:
fato_titulos.filter(F.col("status") == "aberto").select("status", "data_vencimento", "data_pagamento", "dias_atraso").show(10)

fato_titulos.groupBy("situacao").agg(F.sum("valor").alias("valor_total")).show()

fato_titulos.groupBy("faixa_aging").count().show()